# 07 — Teste do classificador BERTimbau

Notebook de **smoke test**: carrega o modelo salvo em `modelo_bertimbau/` e classifica
algumas ementas de exemplo. Serve para confirmar que o ambiente está OK (torch, transformers,
tokenizer e pasta do modelo) **antes** de subir a app Streamlit.

- Escolhe **GPU automaticamente** se houver kernel compatível; senão cai para **CPU** (fallback seguro).
- Não treina nada — só inferência. Roda em CPU tranquilamente.

> Pré-requisitos na raiz do projeto: `modelo_bertimbau/`, `proposicoes_temas.csv`, `resultados_bertimbau.csv`.

In [1]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.preprocessing import MultiLabelBinarizer

print('torch', torch.__version__)

c:\Users\gutob\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.11.0+cu128


## 1) Conferir a pasta do modelo

Os arquivos têm que estar **direto** em `modelo_bertimbau/` (não aninhados em
`modelo_bertimbau/modelo_bertimbau/`). A célula abaixo detecta esse erro comum.

In [2]:
MODELO_DIR = 'modelo_bertimbau'
p = Path(MODELO_DIR)
necessarios = ['config.json', 'tokenizer.json', 'model.safetensors']
faltando = [f for f in necessarios if not (p / f).exists()]
if faltando:
    aninhado = p / 'modelo_bertimbau'
    if aninhado.exists() and (aninhado / 'config.json').exists():
        raise SystemExit(
            f'Arquivos aninhados em {aninhado}. Mova o conteudo para {p}/ (ver README).')
    raise SystemExit(f'Faltam arquivos em {MODELO_DIR}/: {faltando}. Baixe a pasta do Drive.')
print('Pasta do modelo OK ->', sorted(f.name for f in p.iterdir())[:6])

Pasta do modelo OK -> ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


## 2) Escolher o device (GPU com fallback seguro)

`torch.cuda.is_available()` pode retornar `True` mas a placa não ter kernel compatível
(ex.: GPU nova com build antigo do torch). Aqui fazemos um **teste real** de execução na GPU;
se falhar, usamos CPU.

In [3]:
def escolher_device():
    if not torch.cuda.is_available():
        return 'cpu', 'CUDA indisponivel (rodando em CPU)'
    try:
        _ = torch.ones(8, device='cuda') @ torch.ones(8, device='cuda')  # forca um kernel
        torch.cuda.synchronize()
        return 'cuda', f'GPU: {torch.cuda.get_device_name(0)}'
    except Exception as e:
        return 'cpu', f'GPU incompativel ({type(e).__name__}); usando CPU'

device, msg = escolher_device()
print(device, '|', msg)

cuda | GPU: NVIDIA GeForce RTX 5050 Laptop GPU


## 3) Carregar tokenizer + modelo

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_DIR)
modelo = AutoModelForSequenceClassification.from_pretrained(MODELO_DIR)
modelo.to(device).eval()
print('Modelo carregado em', device)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11035.91it/s]


Modelo carregado em cuda


## 4) Temas e limiares

A ordem dos 32 temas é a mesma do treino (`classes_` do `MultiLabelBinarizer`). O limiar de
decisão é **por tema** (ajustado na validação, em `resultados_bertimbau.csv`).

In [5]:
props = pd.read_csv('dados/proposicoes_temas.csv', dtype=str)
nomes_temas = list(MultiLabelBinarizer().fit(props['temas'].str.split('|')).classes_)

resb = pd.read_csv('dados/resultados_bertimbau.csv')
lim_map = dict(zip(resb['tema'], resb['limiar']))
limiares = np.array([float(lim_map.get(t, 0.5)) for t in nomes_temas])
print(len(nomes_temas), 'temas carregados')

32 temas carregados


## 5) Função de classificação

In [6]:
@torch.no_grad()
def classificar(ementa, top=8):
    enc = tokenizer(ementa.strip(), truncation=True, max_length=192,
                    return_tensors='pt').to(device)
    probs = torch.sigmoid(modelo(**enc).logits)[0].cpu().numpy()
    decididos = [nomes_temas[j] for j in np.argsort(-probs) if probs[j] >= limiares[j]]
    ranking = [(nomes_temas[j], float(probs[j])) for j in np.argsort(-probs)[:top]]
    return decididos, ranking

## 6) Testar com exemplos reais da base

Pegamos uma ementa real de alguns temas e mostramos os temas previstos + as maiores
probabilidades. Se isso roda sem erro e os temas fazem sentido, o ambiente está OK.

In [7]:
temas_demo = ['Saúde', 'Educação', 'Meio Ambiente e Desenvolvimento Sustentável',
              'Direito Penal e Processual Penal', 'Esporte e Lazer']
exemplos = []
for t in temas_demo:
    r = props[props['temas'].fillna('').str.contains(t, regex=False)].head(1)
    if len(r):
        exemplos.append((t, r.iloc[0]['ementa']))

for tema_real, ementa in exemplos:
    decididos, ranking = classificar(ementa)
    print('=' * 80)
    print('Tema esperado :', tema_real)
    print('Ementa        :', ementa[:140], '...')
    print('Temas previstos:', '  |  '.join(decididos) if decididos else '(nenhum passou do limiar)')
    print('Top probabilidades:')
    for nome, prob in ranking[:5]:
        print(f'   {prob:5.2f}  {nome}')

Tema esperado : Saúde
Ementa        : Dispõe sobre a prioridade epidemiológica no tratamento de doenças neuromusculares com paralisia motora e dá outras providências. ...
Temas previstos: Saúde
Top probabilidades:
    0.99  Saúde
    0.02  Direitos Humanos e Minorias
    0.01  Administração Pública
    0.01  Educação
    0.01  Finanças Públicas e Orçamento
Tema esperado : Educação
Ementa        : Altera o art. 26 da Lei nº 9.394, de 20 de dezembro de 1996 (Lei de Diretrizes e Bases da Educação Nacional) ...
Temas previstos: Educação
Top probabilidades:
    0.99  Educação
    0.09  Direitos Humanos e Minorias
    0.02  Saúde
    0.01  Defesa e Segurança
    0.01  Finanças Públicas e Orçamento
Tema esperado : Meio Ambiente e Desenvolvimento Sustentável
Ementa        : Estabelece redução de tributos para produtos adequados à economia verde de baixo carbono.  NOVA EMENTA: Institui o Sistema Brasileiro de Com ...
Temas previstos: Meio Ambiente e Desenvolvimento Sustentável  |  Finanças Públ

## 7) Testar com um texto livre

In [8]:
texto = ('Institui o Programa Nacional de Incentivo a vacinação infantil e amplia a oferta '
         'de imunizantes na rede publica de saude.')
decididos, ranking = classificar(texto)
print('Texto:', texto)
print('Temas previstos:', '  |  '.join(decididos) if decididos else '(nenhum passou do limiar)')
print('Top probabilidades:')
for nome, prob in ranking[:5]:
    print(f'   {prob:5.2f}  {nome}')

print('\nOK: classificador funcionando em', device)

Texto: Institui o Programa Nacional de Incentivo a vacinação infantil e amplia a oferta de imunizantes na rede publica de saude.
Temas previstos: Saúde  |  Direitos Humanos e Minorias
Top probabilidades:
    0.98  Saúde
    0.50  Direitos Humanos e Minorias
    0.03  Administração Pública
    0.01  Educação
    0.00  Finanças Públicas e Orçamento

OK: classificador funcionando em cuda
